Aggregate Customer Revenue

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

silver_df = spark.table(
    "retail_analytics_catalog.silver.sales_clean"
)

customer_sales = (
    silver_df
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        sum("sales_amount")
        .alias("total_sales")
    )
)

Ranking

In [0]:
window_spec = Window.orderBy(
    desc("total_sales")
)

top_customers = (
    customer_sales
    .withColumn(
        "rank",
        dense_rank().over(window_spec)
    )
)

In [0]:
top_customers.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"retail_analytics_catalog.gold.top_customers"
)